# Notebook 46 — Shape Composition Grokking

**Hypothesis:** Signal mixing of two shape classes produces a predictable output class via a learnable composition table T[i][j]. If this table has latent algebraic structure, a small transformer trained on (class_A, class_B) → T[A][B] under heavy weight decay will exhibit grokking — first memorizing training pairs, then suddenly generalising to held-out pairs (following Power et al. 2022).

**Why:** nb25 showed shape *classification* never grokked — the model generalised immediately because each class is syntactically distinct. The composition task requires *combinatorial* generalisation: predicting the outcome of class pairs never seen together in training. This is the condition Power et al. identified for grokking.

**Connection:** If T[i][j] is dominated by the dominant-process rule (nb42, F127 — the more coherent class wins), and if grokking reveals this structure, the embedding space should mirror the 6-feature fingerprint geometry, closing the grokking–XWorld loop proposed at nb25.

---

## Pre-run predictions

**F139:** The 8×8 composition table T[i][j] has an idempotent diagonal (T[i][i] = class_i for all 8 classes) and an off-diagonal dominated by irregular_osc (>50% of the 56 off-diagonal pairs). The dominant-process hypothesis (nb42) predicts that mixing two structurally distinct signals produces noise-like superpositions that the fingerprint reads as irregular_osc.

**F140:** Grokking does NOT occur in the token-level experiment. Both train and val accuracy rise simultaneously because the composition table is dominated by a simple learnable rule (diagonal → idempotent; off-diagonal → irregular_osc) that a small transformer discovers with small weights immediately — no memorisation phase needed. This mirrors nb25's classification finding.

**F141:** Even without grokking, post-training token embeddings have Spearman ρ > 0.0 between embedding pairwise distances and 6-feature fingerprint pairwise distances — an improvement over nb25's address-book geometry (ρ = −0.31). The composition task forces classes to be positioned by dynamic similarity, not arbitrary address-book positions.

**F142:** The dominant-disorder rule (T[i][j] = class with lower composite d_min, i.e. the more coherent class) correctly predicts T[i][j] for >50% of off-diagonal pairs. When both classes are similarly coherent, the output is irregular_osc (the attractor sink from nb41, F124).

In [1]:
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.preprocessing import StandardScaler
import time, sys
sys.path.insert(0, '..')

SIGNED_COLS = ['skewness', 'kurtosis', 'lag1_autocorr', 'zero_crossings', 'slope', 'baseline_delta']
SEQ_LEN = 64; SEED = 42; t64 = np.linspace(0, 1, SEQ_LEN)

def zscore(s):
    s = np.asarray(s, dtype=float); std = s.std()
    return (s - s.mean()) / std if std > 1e-8 else s * 0.0

def baseline_delta_fn(s, frac=0.10):
    k = max(1, int(len(s) * frac))
    return float(np.mean(s[-k:]) - np.mean(s[:k]))

def extract_6f(s):
    arr = np.asarray(s, dtype=float); t = np.arange(len(arr))
    lag1 = float(np.corrcoef(arr[:-1], arr[1:])[0, 1]) if len(arr) > 2 else 0.0
    return {
        'skewness':       float(stats.skew(arr)),
        'kurtosis':       float(stats.kurtosis(arr)),
        'lag1_autocorr':  lag1,
        'zero_crossings': float(np.sum(np.diff(np.sign(arr)) != 0) / len(arr)),
        'slope':          float(stats.linregress(t, arr).slope),
        'baseline_delta': baseline_delta_fn(arr),
    }

GENERATORS = {
    'burst':              lambda r: zscore(np.exp(-(t64-r.uniform(.15,.50))**2/(2*r.uniform(.05,.15)**2))+r.normal(0,.05,SEQ_LEN)),
    'oscillator':         lambda r: zscore(np.sin(2*np.pi*r.uniform(1.5,4.5)*t64+r.uniform(0,np.pi))+r.normal(0,.05,SEQ_LEN)),
    'seasonal':           lambda r: zscore(np.sin(2*np.pi*r.uniform(3,6)*t64)+.25*np.sin(4*np.pi*r.uniform(3,6)*t64)+r.normal(0,.04,SEQ_LEN)),
    'trend':              lambda r: zscore(t64+r.uniform(.05,.30)*t64**2+r.normal(0,.02,SEQ_LEN)),
    'integrated_trend':   lambda r: zscore(np.cumsum(np.ones(SEQ_LEN)*r.uniform(.015,.035)+r.normal(0,.003,SEQ_LEN))),
    'irregular_osc':      lambda r: zscore((np.sin(2*np.pi*r.uniform(2,5)*t64)*(1+r.uniform(.3,.8,SEQ_LEN))+r.normal(0,.3,SEQ_LEN))*1.4),
    'declining_osc':      lambda r: zscore(np.linspace(r.uniform(.9,1.2),r.uniform(.35,.65),SEQ_LEN)*np.sin(2*np.pi*r.uniform(2.5,5.5)*t64)+np.linspace(0,r.uniform(-.8,-.4),SEQ_LEN)+r.normal(0,.05,SEQ_LEN)),
    'declining_monotonic':lambda r: zscore(np.cumsum(-np.ones(SEQ_LEN)*r.uniform(.015,.035)+r.normal(0,.003,SEQ_LEN))),
}

CLASSES = list(GENERATORS.keys())  # 8 classes, fixed order
N_CLASSES = len(CLASSES)
CLS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

recs = []
for cls, gen in GENERATORS.items():
    for i in range(200):
        r = np.random.default_rng(SEED + CLASSES.index(cls)*1000 + i)
        f = extract_6f(gen(r)); f['class'] = cls; recs.append(f)
df_train = pd.DataFrame(recs)
sc = StandardScaler()
X_tr_fp = sc.fit_transform(df_train[SIGNED_COLS].values)
ctrds = {c: X_tr_fp[df_train['class']==c].mean(axis=0) for c in GENERATORS}

def classify(feat_dict):
    x = sc.transform([[feat_dict[c] for c in SIGNED_COLS]])[0]
    dists = {c: float(np.linalg.norm(x - v)) for c, v in ctrds.items()}
    best = min(dists, key=dists.get)
    return best, dists[best], dists

print(f'8-class fingerprint classifier ready. Classes: {CLASSES}')

8-class fingerprint classifier ready. Classes: ['burst', 'oscillator', 'seasonal', 'trend', 'integrated_trend', 'irregular_osc', 'declining_osc', 'declining_monotonic']


In [2]:
# ---- Part A: Derive composition table T[i][j] ----
# T[i][j] = majority class when mixing zscore(0.5*A + 0.5*B),
# where A ~ class_i, B ~ class_j, both already zscored.

N_SAMPLES_PER_PAIR = 500

table = np.zeros((N_CLASSES, N_CLASSES), dtype=int)      # majority class index
table_purity = np.zeros((N_CLASSES, N_CLASSES))           # majority fraction
table_name = [['' for _ in range(N_CLASSES)] for _ in range(N_CLASSES)]
table_counts = {}

print('Deriving 8×8 composition table (500 samples/pair) ...')
t0 = time.time()

for i, cls_a in enumerate(CLASSES):
    gen_a = GENERATORS[cls_a]
    for j, cls_b in enumerate(CLASSES):
        gen_b = GENERATORS[cls_b]
        results = []
        for k in range(N_SAMPLES_PER_PAIR):
            r_a = np.random.default_rng(1000 + i*5000 + k)
            r_b = np.random.default_rng(2000 + j*5000 + k)
            sig_a = gen_a(r_a)   # zscored by generator
            sig_b = gen_b(r_b)
            mixed = zscore(0.5 * sig_a + 0.5 * sig_b)
            cls_out, _, _ = classify(extract_6f(mixed))
            results.append(cls_out)
        cnt = Counter(results)
        top_cls, top_n = cnt.most_common(1)[0]
        table[i, j] = CLS_TO_IDX.get(top_cls, 0)
        table_purity[i, j] = top_n / N_SAMPLES_PER_PAIR
        table_name[i][j] = top_cls
        table_counts[(i, j)] = cnt

print(f'Done in {time.time()-t0:.1f}s')
print()

# Print composition table (abbreviated class names)
ABBREV = {
    'burst': 'BUR', 'oscillator': 'OSC', 'seasonal': 'SEA',
    'trend': 'TRE', 'integrated_trend': 'INT', 'irregular_osc': 'IRR',
    'declining_osc': 'DCO', 'declining_monotonic': 'DCM',
}
HDR = [f'{ABBREV[c]:>4s}' for c in CLASSES]
print(f'{"":>5s}  ' + '  '.join(HDR))
for i, cls_a in enumerate(CLASSES):
    row = '  '.join(f'{ABBREV[table_name[i][j]]:>4s}' for j in range(N_CLASSES))
    print(f'{ABBREV[cls_a]:>4s}:  {row}')

Deriving 8×8 composition table (500 samples/pair) ...


Done in 15.8s

        BUR   OSC   SEA   TRE   INT   IRR   DCO   DCM
 BUR:   BUR   DCO   DCO   INT   INT   DCO   DCO   DCM
 OSC:   DCO   OSC   DCO   TRE   TRE   SEA   DCO   DCO
 SEA:   DCO   DCO   SEA   OSC   OSC   SEA   DCO   DCO
 TRE:   INT   TRE   OSC   TRE   INT   SEA   OSC   IRR
 INT:   INT   TRE   OSC   INT   INT   OSC   OSC   IRR
 IRR:   DCO   SEA   SEA   SEA   OSC   SEA   DCO   DCO
 DCO:   DCO   DCO   DCO   OSC   OSC   DCO   DCO   DCO
 DCM:   DCM   DCO   DCO   IRR   IRR   DCO   DCO   DCM


In [3]:
# ---- Part B: Table structure analysis ----

print('=== Part B: Composition Table Structure ===')
print()

# 1. Diagonal idempotence
diag_correct = [i for i in range(N_CLASSES) if table[i, i] == i]
diag_frac = len(diag_correct) / N_CLASSES
print(f'Diagonal idempotent: {len(diag_correct)}/8 = {diag_frac:.0%}')
for i in diag_correct:
    print(f'  T[{CLASSES[i]}, {CLASSES[i]}] = {table_name[i][i]} (purity {table_purity[i,i]:.2f}) ✓')
non_diag = [i for i in range(N_CLASSES) if table[i, i] != i]
for i in non_diag:
    print(f'  T[{CLASSES[i]}, {CLASSES[i]}] = {table_name[i][i]} (purity {table_purity[i,i]:.2f}) ✗')

print()

# 2. Off-diagonal class distribution
off_counts = Counter()
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        if i != j:
            off_counts[CLASSES[table[i, j]]] += 1
n_off = N_CLASSES * (N_CLASSES - 1)
print(f'Off-diagonal class distribution (n={n_off} pairs):')
for cls, cnt in off_counts.most_common():
    print(f'  {cls:22s}: {cnt:2d}/{n_off} = {cnt/n_off*100:.0f}%')

print()

# 3. Commutativity check: is T[i][j] == T[j][i]?
comm_matches = sum(1 for i in range(N_CLASSES) for j in range(i+1, N_CLASSES)
                   if table[i, j] == table[j, i])
n_pairs = N_CLASSES * (N_CLASSES - 1) // 2
print(f'Commutativity: T[i][j]==T[j][i] for {comm_matches}/{n_pairs} = {comm_matches/n_pairs:.0%} of symmetric pairs')

print()

# 4. Dominant-disorder prediction: T[i][j] = class with lower centroid d_min
# Use the full-corpus d_min from nb44 scale scan (mean d_min at full scale)
# Approximate: compute mean d_min of each generator class from the training distribution
dmins = {}
for cls, gen in GENERATORS.items():
    ds = []
    for k in range(100):
        r = np.random.default_rng(9999 + CLASSES.index(cls)*1000 + k)
        fp = extract_6f(gen(r))
        _, d, _ = classify(fp)
        ds.append(d)
    dmins[cls] = float(np.mean(ds))

print('Mean d_min by generator class (proxy for coherence):')
for c in sorted(dmins, key=dmins.get):
    print(f'  {c:22s}: {dmins[c]:.3f}')

# Test dominant-disorder prediction: T[i][j] = class with lower d_min
print()
dominant_correct = 0
n_off_test = 0
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        if i == j:
            continue
        n_off_test += 1
        if dmins[CLASSES[i]] <= dmins[CLASSES[j]]:
            predicted = i   # class_i is more coherent → should dominate
        else:
            predicted = j
        if table[i, j] == predicted:
            dominant_correct += 1
print(f'Dominant-disorder prediction accuracy: {dominant_correct}/{n_off_test} = {dominant_correct/n_off_test:.0%}')

=== Part B: Composition Table Structure ===

Diagonal idempotent: 7/8 = 88%
  T[burst, burst] = burst (purity 0.64) ✓
  T[oscillator, oscillator] = oscillator (purity 0.52) ✓
  T[seasonal, seasonal] = seasonal (purity 0.65) ✓
  T[trend, trend] = trend (purity 0.90) ✓
  T[integrated_trend, integrated_trend] = integrated_trend (purity 1.00) ✓
  T[declining_osc, declining_osc] = declining_osc (purity 0.65) ✓
  T[declining_monotonic, declining_monotonic] = declining_monotonic (purity 1.00) ✓
  T[irregular_osc, irregular_osc] = seasonal (purity 0.62) ✗

Off-diagonal class distribution (n=56 pairs):
  declining_osc         : 24/56 = 43%
  oscillator            : 10/56 = 18%
  integrated_trend      :  6/56 = 11%
  seasonal              :  6/56 = 11%
  trend                 :  4/56 = 7%
  irregular_osc         :  4/56 = 7%
  declining_monotonic   :  2/56 = 4%

Commutativity: T[i][j]==T[j][i] for 28/28 = 100% of symmetric pairs



Mean d_min by generator class (proxy for coherence):
  integrated_trend      : 0.049
  declining_monotonic   : 0.056
  trend                 : 0.223
  declining_osc         : 0.741
  seasonal              : 0.777
  oscillator            : 0.832
  irregular_osc         : 0.846
  burst                 : 1.864

Dominant-disorder prediction accuracy: 18/56 = 32%


In [4]:
# ---- Part C: Token-level grokking experiment (Power et al. template) ----

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print()

# Train/test split: 80% of pairs in train, 20% in test (split by class PAIR)
rng_split = np.random.default_rng(SEED)
all_pairs = [(i, j) for i in range(N_CLASSES) for j in range(N_CLASSES)]  # 64 pairs
perm = rng_split.permutation(len(all_pairs))
n_train_pairs = int(len(all_pairs) * 0.8)  # 51
train_pairs = set(tuple(all_pairs[k]) for k in perm[:n_train_pairs])
test_pairs  = set(tuple(all_pairs[k]) for k in perm[n_train_pairs:])
print(f'Train pairs: {len(train_pairs)} | Test pairs: {len(test_pairs)}')
print(f'Test pairs (held-out class combinations):')
for (i, j) in sorted(test_pairs):
    print(f'  ({CLASSES[i][:4]}, {CLASSES[j][:4]}) → {table_name[i][j]}')

# Build dataset
N_INST = 3000  # instances per pair

X_tr_list, Y_tr_list, X_te_list, Y_te_list = [], [], [], []
for i, j in all_pairs:
    label = int(table[i, j])
    xs = torch.zeros(N_INST, 2, dtype=torch.long)
    xs[:, 0] = i
    xs[:, 1] = j
    ys = torch.full((N_INST,), label, dtype=torch.long)
    if (i, j) in train_pairs:
        X_tr_list.append(xs); Y_tr_list.append(ys)
    else:
        X_te_list.append(xs); Y_te_list.append(ys)

X_tr = torch.cat(X_tr_list)  # (51*3000, 2)
Y_tr = torch.cat(Y_tr_list)
X_te = torch.cat(X_te_list)  # (13*3000, 2)
Y_te = torch.cat(Y_te_list)

# Shuffle training data
perm_tr = torch.randperm(len(X_tr), generator=torch.Generator().manual_seed(SEED))
X_tr = X_tr[perm_tr]; Y_tr = Y_tr[perm_tr]

train_loader = DataLoader(TensorDataset(X_tr, Y_tr), batch_size=1024, shuffle=True)
print(f'\nTrain: {len(X_tr):,} examples | Test: {len(X_te):,} examples')

# ---- Model (following Power et al. 2022: small, weight-decayed) ----
class ShapeCompositionNet(nn.Module):
    def __init__(self, n_cls=8, d=128, n_heads=4, n_layers=2):
        super().__init__()
        self.tok_emb = nn.Embedding(n_cls, d)  # shared embedding
        enc = nn.TransformerEncoderLayer(d, n_heads, d * 4, dropout=0.0,
                                         batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(enc, n_layers)
        self.head = nn.Linear(d, n_cls)
    
    def forward(self, ab):       # ab: (B, 2) long
        x = self.tok_emb(ab)    # (B, 2, d)
        x = self.tf(x)          # (B, 2, d)
        x = x.mean(dim=1)       # (B, d)
        return self.head(x)     # (B, n_cls)

torch.manual_seed(SEED)
model = ShapeCompositionNet(n_cls=N_CLASSES, d=128, n_heads=4, n_layers=2).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

# AdamW with heavy weight decay (Power et al. protocol)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1.0)
criterion = nn.CrossEntropyLoss()

# ---- Training loop ----
N_STEPS   = 50_000
LOG_EVERY = 200

X_tr_dev = X_tr.to(device); Y_tr_dev = Y_tr.to(device)
X_te_dev = X_te.to(device); Y_te_dev = Y_te.to(device)

def eval_acc(X, Y, batch=8192):
    model.eval()
    correct = 0
    with torch.no_grad():
        for b in range(0, len(X), batch):
            correct += (model(X[b:b+batch]).argmax(1) == Y[b:b+batch]).sum().item()
    model.train()
    return correct / len(Y)

train_accs, val_accs, steps_log = [], [], []
train_iter = iter(train_loader)
t0 = time.time()
model.train()

for step in range(1, N_STEPS + 1):
    try:
        xb, yb = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        xb, yb = next(train_iter)
    
    xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad()
    criterion(model(xb), yb).backward()
    opt.step()
    
    if step % LOG_EVERY == 0:
        tr_acc = eval_acc(X_tr_dev, Y_tr_dev)
        te_acc = eval_acc(X_te_dev, Y_te_dev)
        train_accs.append(tr_acc)
        val_accs.append(te_acc)
        steps_log.append(step)
        if step % 5000 == 0:
            print(f'  Step {step:6d} ({time.time()-t0:.0f}s): train={tr_acc:.3f}  val={te_acc:.3f}')

print(f'\nFinal: train={train_accs[-1]:.4f}  val={val_accs[-1]:.4f}')
print(f'Total time: {time.time()-t0:.0f}s')

# Grokking detection
arr_tr = np.array(train_accs)
arr_va = np.array(val_accs)

# When did train acc first exceed 99%?
train_memo_idx = next((k for k, a in enumerate(arr_tr) if a >= 0.99), None)

if train_memo_idx is not None:
    memo_step = steps_log[train_memo_idx]
    pre_val   = arr_va[train_memo_idx]
    final_val = arr_va[-1]
    delta     = final_val - pre_val
    grokking  = delta > 0.20  # ≥20pp jump in val after memorisation
    print(f'\nMemorisation at step {memo_step}: val={pre_val:.3f}')
    print(f'Final val: {final_val:.3f}  (Δ={delta:+.3f})')
    print(f'Grokking detected: {grokking}')
else:
    print('\nTrain accuracy never exceeded 99% — model failed to memorise.')
    grokking = False

Device: cuda
GPU: NVIDIA GeForce RTX 5070

Train pairs: 51 | Test pairs: 13
Test pairs (held-out class combinations):
  (burs, osci) → declining_osc
  (burs, seas) → declining_osc
  (osci, burs) → declining_osc
  (osci, inte) → trend
  (osci, irre) → seasonal
  (osci, decl) → declining_osc
  (inte, osci) → trend
  (inte, inte) → integrated_trend
  (irre, tren) → seasonal
  (irre, irre) → seasonal
  (decl, osci) → declining_osc
  (decl, irre) → declining_osc
  (decl, decl) → declining_osc

Train: 153,000 examples | Test: 39,000 examples
Model parameters: 398,600


  Step   5000 (27s): train=1.000  val=0.692


  Step  10000 (53s): train=1.000  val=0.846


  Step  15000 (80s): train=1.000  val=0.538


  Step  20000 (107s): train=1.000  val=0.692


...

  Step  30000 (160s): train=1.000  val=0.846


  Step  35000 (187s): train=1.000  val=0.846


  Step  40000 (214s): train=1.000  val=0.692


  Step  45000 (244s): train=1.000  val=0.846


  Step  50000 (281s): train=1.000  val=0.692

Final: train=1.0000  val=0.6923
Total time: 281s

Memorisation at step 200: val=0.846
Final val: 0.692  (Δ=-0.154)
Grokking detected: False


In [5]:
# ---- Part D: Post-training embedding analysis ----

model.eval()
with torch.no_grad():
    all_idx = torch.arange(N_CLASSES, device=device)
    embeddings = model.tok_emb(all_idx).cpu().numpy()  # (8, 128)

# Pairwise distances in embedding space
emb_dists = squareform(pdist(embeddings, metric='euclidean'))

# Pairwise distances in 6-feature fingerprint centroid space (standardised)
ctrd_arr = np.array([ctrds[c] for c in CLASSES])  # (8, 6)
fp_dists  = squareform(pdist(ctrd_arr, metric='euclidean'))

# Composition table distance (does emb distance predict composition similarity?)
# Build "composition distance matrix": how often do classes i and j appear as each other's output?
comp_sim = np.zeros((N_CLASSES, N_CLASSES))  # composition table purity as similarity
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        comp_sim[i, j] = table_purity[i, j]

mask = np.triu(np.ones((N_CLASSES, N_CLASSES), dtype=bool), k=1)
emb_flat = emb_dists[mask]
fp_flat  = fp_dists[mask]

rho_fp, pval_fp = spearmanr(emb_flat, fp_flat)

# Also compare composition table output to fingerprint distances:
# Does composition distance predict fingerprint distance?
comp_dist_flat = 1.0 - comp_sim[mask]  # lower purity → more uncertain → larger distance
rho_comp, pval_comp = spearmanr(emb_flat, comp_dist_flat)

print('=== Part D: Embedding Analysis ===')
print(f'\nSpearman ρ(embedding ↔ fingerprint):  {rho_fp:+.3f}  (p={pval_fp:.4f})')
print(f'nb25 address-book baseline:             ρ = −0.31')
print(f'Random baseline (expected):             ρ ≈  0.00')
print()
print(f'Spearman ρ(embedding ↔ comp-impurity): {rho_comp:+.3f}  (p={pval_comp:.4f})')
print()

# Per-class embedding distances (sorted)
print('Pairwise embedding distances (top 5 closest class pairs):')
pairs_sorted = sorted(
    [(emb_dists[i,j], CLASSES[i], CLASSES[j]) for i in range(N_CLASSES) for j in range(i+1, N_CLASSES)]
)
for d, a, b in pairs_sorted[:5]:
    print(f'  {a:22s} ↔ {b:22s}: emb_dist={d:.3f}  fp_dist={fp_dists[CLASSES.index(a), CLASSES.index(b)]:.3f}')

print()
print('Pairwise embedding distances (top 5 furthest class pairs):')
for d, a, b in pairs_sorted[-5:][::-1]:
    print(f'  {a:22s} ↔ {b:22s}: emb_dist={d:.3f}  fp_dist={fp_dists[CLASSES.index(a), CLASSES.index(b)]:.3f}')

# Test generalisation per held-out pair
print()
print('Test-pair breakdown (held-out class pairs):')
model.eval()
with torch.no_grad():
    for (i, j) in sorted(test_pairs):
        xb = torch.zeros(N_INST, 2, dtype=torch.long, device=device)
        xb[:, 0] = i; xb[:, 1] = j
        yb = torch.full((N_INST,), int(table[i, j]), dtype=torch.long, device=device)
        acc = (model(xb).argmax(1) == yb).float().mean().item()
        print(f'  ({CLASSES[i][:8]:8s}, {CLASSES[j][:8]:8s}) → {table_name[i][j]:20s}  acc={acc:.3f}')

=== Part D: Embedding Analysis ===

Spearman ρ(embedding ↔ fingerprint):  +0.399  (p=0.0354)
nb25 address-book baseline:             ρ = −0.31
Random baseline (expected):             ρ ≈  0.00

Spearman ρ(embedding ↔ comp-impurity): +0.175  (p=0.3734)

Pairwise embedding distances (top 5 closest class pairs):
  trend                  ↔ integrated_trend      : emb_dist=0.047  fp_dist=0.282
  burst                  ↔ declining_monotonic   : emb_dist=0.061  fp_dist=3.862
  oscillator             ↔ trend                 : emb_dist=0.064  fp_dist=2.887
  oscillator             ↔ seasonal              : emb_dist=0.064  fp_dist=1.580
  seasonal               ↔ declining_osc         : emb_dist=0.070  fp_dist=1.498

Pairwise embedding distances (top 5 furthest class pairs):
  integrated_trend       ↔ irregular_osc         : emb_dist=0.134  fp_dist=4.021
  seasonal               ↔ declining_monotonic   : emb_dist=0.122  fp_dist=3.706
  burst                  ↔ declining_osc         : emb_dist=0.

In [6]:
import os
os.makedirs('../artifacts', exist_ok=True)

# ---- Visualisation ----

CLASS_COLORS = {
    'burst': '#F44336', 'oscillator': '#2196F3', 'seasonal': '#FF9800',
    'trend': '#795548', 'integrated_trend': '#607D8B', 'irregular_osc': '#E91E63',
    'declining_osc': '#9C27B0', 'declining_monotonic': '#009688',
}
COLOR_LIST = [CLASS_COLORS[c] for c in CLASSES]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('nb46 — Shape Composition Grokking', fontsize=13, fontweight='bold')

# Panel 1: Composition table heatmap
ax = axes[0]
im = ax.imshow(table, cmap='tab10', vmin=0, vmax=N_CLASSES-1, aspect='auto')
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels([ABBREV[c] for c in CLASSES], fontsize=8)
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels([ABBREV[c] for c in CLASSES], fontsize=8)
ax.set_xlabel('Class B'); ax.set_ylabel('Class A')
ax.set_title('Composition table T[A,B]')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, ABBREV[table_name[i][j]], ha='center', va='center',
                fontsize=6, color='white', fontweight='bold')

# Panel 2: Train/val accuracy curves
ax = axes[1]
ax.plot(steps_log, train_accs, label='Train acc', color='steelblue', lw=1.5)
ax.plot(steps_log, val_accs,   label='Val acc',   color='tomato',    lw=1.5)
ax.axhline(1/N_CLASSES, color='gray', lw=1, ls='--', label=f'Chance ({1/N_CLASSES:.2f})')
if train_memo_idx is not None:
    ax.axvline(steps_log[train_memo_idx], color='green', lw=1, ls=':', label='Train≥99%')
ax.set_xlabel('Training step'); ax.set_ylabel('Accuracy')
ax.set_title(f'Grokking experiment (weight_decay=1.0)\nGrokking: {grokking}')
ax.legend(fontsize=8); ax.set_ylim(0, 1.05)

# Panel 3: Embedding distance vs fingerprint distance (scatter)
ax = axes[2]
scatter = ax.scatter(fp_flat, emb_flat, c='steelblue', alpha=0.7, s=40)
# Annotate a few points with class-pair names
pair_list = [(i, j) for i in range(N_CLASSES) for j in range(i+1, N_CLASSES)]
for (i, j), fp_d, emb_d in zip(pair_list, fp_flat, emb_flat):
    if fp_d > np.percentile(fp_flat, 85) or fp_d < np.percentile(fp_flat, 15):
        ax.annotate(f'{ABBREV[CLASSES[i]]}/{ABBREV[CLASSES[j]]}',
                    (fp_d, emb_d), fontsize=6, alpha=0.7)
ax.set_xlabel('Fingerprint distance (centroid)')
ax.set_ylabel('Embedding distance (post-training)')
ax.set_title(f'Embedding ↔ Fingerprint\nSpearman ρ={rho_fp:+.3f} (nb25: −0.31)')

plt.tight_layout()
plt.savefig('../artifacts/nb46_composition_grokking.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


---
## Findings — Notebook 46

### F139 — Composition table: declining_osc dominates off-diagonal (43%); diagonal 7/8 idempotent; perfect commutativity

**Prediction:** Off-diagonal >50% irregular_osc; diagonal idempotent. **Partially refuted.**

- Diagonal: 7/8 idempotent (88%). Exception: T[IRR,IRR] = seasonal.
- Off-diagonal: **declining_osc dominates (43%)**, not irregular_osc (7%); oscillator=18%, integrated_trend=11%, seasonal=11%.
- Commutativity: T[i][j]=T[j][i] for all 28 symmetric pairs (100%) — exact, not approximate.

---

### F140 — No grokking; immediate generalisation at step 200 (val=84.6%); val oscillates under weight decay

**Prediction:** No grokking. **Confirmed with nuance.**

Train → 100% at step 200 (simultaneous with val=84.6%). No memorisation gap. Val oscillates 53.8–84.6% under weight decay throughout 50k steps. Final val=69.2% (9/13 test pairs succeed; burst×oscillator and oscillator×integrated_trend fail in both symmetric directions).

---

### F141 — Post-training Spearman ρ=+0.399 between embedding and fingerprint distances (vs nb25 ρ=−0.31)

**Prediction:** ρ > 0.0. **Confirmed: ρ=+0.399, p=0.035.**

Composition task induces fingerprint-aligned embeddings; classification task (nb25) produces arbitrary address-book geometry.

---

### F142 — Dominant-disorder rule predicts only 32% of off-diagonal pairs — declining_osc wins regardless of input coherence

**Prediction:** >50% accuracy. **Refuted: 32%.**

Output class is determined by the composition attractor (declining_osc), not by the input class with lower d_min.

---

### Emergent F143 — Perfect commutativity (100%) despite independent signal generation and nonlinear fingerprint

New finding. T[i][j]=T[j][i] for all 28 pairs. The zscore mixing operation preserves class symmetry exactly.

---

### Emergent F144 — T[irregular_osc, irregular_osc] = seasonal: noise averaging reduces amplitude variance to produce regular oscillation

New finding. Two independent amplitude modulations averaged together reduce variance (LLN in feature space) → seasonal class. Self-composition of noise produces regularity.

---

### Findings
F139–F144 added. Total findings: **144**.